# 23 · Checking the "stays in the top 10" numbers

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

The final lists say how often each player stayed in the top 10 when the ranking was rebuilt 1,000 times on reshuffled matches (notebook 12). A single bug there would mislead every reader, so this notebook rebuilds those numbers from scratch with different code and a different random seed (200 rebuilds per side).

**Result.** No fault found. The shares add up to exactly 10, as they must. 36 of the 40 players are within the expected random wobble of 200 rebuilds, the average difference is +0.2 points, and every "close call" label stays the same.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P26 - Independent check of the 'stays in the top 10 in X of 100 rebuilds' numbers (audit; nothing existing is changed).  
  
The ranking step (P10) rebuilt the ranking 1,000 times on matches resampled with replacement. This script re-implements  
that from scratch with different code and a different seed: matches are drawn with pandas, repeated matches get new IDs,  
shrinkage strengths are held at their full-data values, only available players are ranked, and the 60-pressure-ball  
floor applies inside each rebuild. 200 rebuilds per side, sequential, 2 threads.  
If P10 has no bug, every player's share here should match P10's within Monte Carlo error  
(about +/- 7 points at 95% for 200 rebuilds when the share is near 50%).
</details>

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from rrlib.common import load, load_json, load_table, save_table, get_logger, Timer, excluded_reasons
from rrlib import ratings as RL
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p26_bootstrap_audit")
v = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]
EX = set(excluded_reasons())
REPS, SEED2 = 200, 777
groups = {m: g for m, g in v.groupby("match_id", sort=False)}
match_ids = np.array(list(groups))
rows = []
for side in ("bat", "bowl"):
    cfg = chosen[side]
    RL.K_RECORD = {}
    full = RL.build_scores(v, side, cfg, comp)
    kfix = dict(RL.K_RECORD); RL.K_RECORD = None
    pool = full.index[~full.index.isin(EX)]
    hits = pd.Series(0, index=pool, dtype=float)
    rng = np.random.default_rng(SEED2)
    with Timer(log, f"{side}: {REPS} independent rebuilds"):
        for b in range(REPS):
            draw = rng.choice(match_ids, size=len(match_ids), replace=True)
            parts = []
            for j, m in enumerate(draw):
                g = groups[m].copy()
                g["match_id"] = j            # a match drawn twice counts as two different matches
                parts.append(g)
            s = pd.concat(parts, ignore_index=True)
            RL.K_FIXED = kfix
            sc = RL.build_scores(s, side, cfg, comp).score
            RL.K_FIXED = None
            sc = sc[sc.index.isin(pool)].sort_values(ascending=False)
            hits.loc[sc.index[:10]] += 1
    share = hits / REPS
    pub = load_table(f"final_all_{side}").set_index("player_id")
    names = load("players").set_index("player_id").name
    top = pub.sort_values("final_rank").head(20).index
    for pid in top:
        p_pub, p_new = float(pub.loc[pid, "p_top10"]), float(share.get(pid, 0.0))
        se = np.sqrt(max(p_pub * (1 - p_pub), 0.01) / REPS)
        rows.append({"side": side, "final_rank": int(pub.loc[pid, "final_rank"]), "name": names.get(pid),
                     "published_per100": round(100 * p_pub, 1), "independent_per100": round(100 * p_new, 1),
                     "difference": round(100 * (p_new - p_pub), 1), "within_95pct_noise": abs(p_new - p_pub) <= 1.96 * se})
    log.info(f"{side}: shares sum to {share.sum():.3f} (must be 10)")
A = pd.DataFrame(rows)
save_table(A, "bootstrap_audit")
pd.set_option("display.width", 200)
log.info("\n" + A.to_string(index=False))
log.info(f"players within Monte Carlo noise: {int(A.within_95pct_noise.sum())} of {len(A)}; "
         f"largest difference {A.difference.abs().max():.1f} points; mean difference {A.difference.mean():+.1f}")
